# Week 3 — Instrument (Lantern AI)

Milestone 3 of the MKT 282 capstone. MaxDiff cuts last week's 14-item long-list to a **5–6 attribute + price** grid, you **launch the CBC in Sawtooth Discover**, and you run a **30-persona silicon twin** with sealed predictions filed before human fielding closes.

The fielding clock is unforgiving: **live at this milestone buys seven days of data collection.** This notebook is the instrument pack — design, scoring, Discover spec, personas, preregistration.

**Answers the memo needs**

1. **MaxDiff (n ≥ 15).** Which attributes made the conjoint, which died, which borderline calls did judgment make.
2. **Final grid** against Exhibit 8: two sentences per attribute on why it earns its slot.
3. **CBC live in Discover:** 10–12 tasks, 3 concepts + none/dual-response, five human pilots + one AI run-through, pilot-fix log, live link.
4. **Silicon twin (A9):** 30 telemetry personas, three sealed predictions timestamped before human fielding closes.

Paste MaxDiff responses into `Data/maxdiff_responses.csv` (template written below). The working grid is Exhibit 8's canonical design — ready to build in Discover this week — and updates automatically if MaxDiff scores reverse a borderline call.


## 0. Setup


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 20)

DATA_DIR = Path("Data")
FACT_PACK = DATA_DIR / "Lantern_Fact_Pack.xlsx"
MAXDIFF_PATH = DATA_DIR / "maxdiff_responses.csv"
PERSONA_PATH = DATA_DIR / "silicon_twin_personas.csv"
RNG = 42
N_MAU = 400_000
TARGET = "upgrade_intent_click"


## 1. Week 2 long-list → MaxDiff items

Price is **not** in MaxDiff. It is required in the CBC. The 13 non-price items below are what n ≥ 15 respondents pick as most / least important.


In [2]:
longlist = pd.DataFrame(
    [
        ["Usage allowance (Light / Standard / Max)", "Keep", "usage",
         "Clickers average 10.9 sessions/week vs 6.6; Exhibit 8 requires usage"],
        ["No-training guarantee", "Keep", "privacy",
         "privacy_mode_share is the largest Week-2 coefficient (OR ≈ 5 on a full private-mode shift)"],
        ["Vault / on-device sensitive mode", "Keep", "privacy",
         "Clickers upload 6.5 docs/30d vs 3.9; case Vault prototype"],
        ["Life-admin agents with click-to-approve", "Keep", "agents",
         "agent_preview_uses_30d: 5.7 vs 3.8; case 'an assistant that asks permission'"],
        ["Autopilot / recurring tasks", "Candidate", "agents",
         "Case product claim; Exhibit 2 serving cost jumps to $4.20 — a fence"],
        ["Email + calendar integrations", "Candidate", "integrations",
         "Case draft-and-send mail; Google/Microsoft fence on native integrations"],
        ["Full suite (docs, storage, budgeting)", "Candidate", "integrations",
         "doc_uploads_30d; case forms, bills, subscriptions"],
        ["Household seats (up to 5)", "Candidate", "seats",
         "22% of MAU share a device but click less (6.0% vs 10.9%) — test, don't assume"],
        ["Priority / advanced-model access", "Candidate", "usage",
         "p*-eligible users: 13.6 sessions/week vs 4.8 ineligible"],
        ["Form-filling agents (school, insurance)", "Candidate", "agents",
         "Case job-to-be-done; students are 21% of the 124k pool"],
        ["Subscription and bill tracking", "Candidate", "integrations",
         "Case life-admin list; pairs with budgeting in the full suite"],
        ["Trip-planning agent", "Candidate", "agents",
         "Case life-admin list; a concrete agent vs generic answers"],
        ["Referral or household gift seats", "Borderline", "seats",
         "referred_by_friend OR 1.41; 39% of p*-eligible arrived via referral vs 13%"],
    ],
    columns=["item", "week2_status", "maps_to", "trace"],
)
price_row = pd.DataFrame(
    [["Price / month ($6, $12, $20, $28, $36)", "Keep — not MaxDiff'd", "price",
      "Exhibit 8; Firefly failed at $19.99; $20 clone is a Week 5 mandatory benchmark"]],
    columns=longlist.columns,
)
print("CBC always includes price. MaxDiff fields the 13 items below.")
longlist


CBC always includes price. MaxDiff fields the 13 items below.


,item,week2_status,maps_to,trace
0,Usage allowance (Light / Standard / Max),Keep,usage,Clickers average 10.9 sessions/week vs 6.6; Exhibit 8 requires usage
1,No-training guarantee,Keep,privacy,privacy_mode_share is the largest Week-2 coefficient (OR ≈ 5 on a full private-mode shift)
2,Vault / on-device sensitive mode,Keep,privacy,Clickers upload 6.5 docs/30d vs 3.9; case Vault prototype
3,Life-admin agents with click-to-approve,Keep,agents,agent_preview_uses_30d: 5.7 vs 3.8; case 'an assistant that asks permission'
4,Autopilot / recurring tasks,Candidate,agents,Case product claim; Exhibit 2 serving cost jumps to $4.20 — a fence
5,Email + calendar integrations,Candidate,integrations,Case draft-and-send mail; Google/Microsoft fence on native integrations
6,"Full suite (docs, storage, budgeting)",Candidate,integrations,"doc_uploads_30d; case forms, bills, subscriptions"
7,Household seats (up to 5),Candidate,seats,"22% of MAU share a device but click less (6.0% vs 10.9%) — test, don't assume"
8,Priority / advanced-model access,Candidate,usage,p*-eligible users: 13.6 sessions/week vs 4.8 ineligible
9,"Form-filling agents (school, insurance)",Candidate,agents,Case job-to-be-done; students are 21% of the 124k pool


### MaxDiff instrument (12 sets × 4 items)

Each respondent sees 12 screens: “Which of these would matter **most** / **least** when choosing a paid personal AI assistant?” Best-minus-worst counting is enough at n ≥ 15; this is triage, not HB.

Field MaxDiff to the **first 15** of the Week 2 recruits (occupation mix 9 professional / 3 student / 3 other). Same screener: very/somewhat likely to pay in the next 12 months.


In [3]:
ITEMS = longlist["item"].tolist()
rng = np.random.default_rng(RNG)


def balanced_sets(items, n_sets=12, k=4, seed=42):
    r = np.random.default_rng(seed)
    counts = {i: 0 for i in items}
    sets = []
    for _ in range(n_sets):
        ranked = sorted(items, key=lambda i: (counts[i], r.random()))
        chosen = ranked[:k]
        r.shuffle(chosen)
        for i in chosen:
            counts[i] += 1
        sets.append(chosen)
    return sets, pd.Series(counts, name="appearances")


sets, appearances = balanced_sets(ITEMS, n_sets=12, k=4, seed=RNG)
maxdiff_design = pd.DataFrame(sets, columns=["A", "B", "C", "D"])
maxdiff_design.index = pd.RangeIndex(1, len(sets) + 1, name="set")
print("Each item should appear 3–4 times:")
print(appearances.sort_values().to_string())
maxdiff_design


Each item should appear 3–4 times:
Life-admin agents with click-to-approve     3
Autopilot / recurring tasks                 3
Priority / advanced-model access            3
Trip-planning agent                         3
Usage allowance (Light / Standard / Max)    4
No-training guarantee                       4
Vault / on-device sensitive mode            4
Email + calendar integrations               4
Full suite (docs, storage, budgeting)       4
Household seats (up to 5)                   4
Form-filling agents (school, insurance)     4
Subscription and bill tracking              4
Referral or household gift seats            4


,A,B,C,D
set,,,,
1,Autopilot / recurring tasks,No-training guarantee,Priority / advanced-model access,Subscription and bill tracking
2,Vault / on-device sensitive mode,"Full suite (docs, storage, budgeting)",Referral or household gift seats,Usage allowance (Light / Standard / Max)
3,Household seats (up to 5),Life-admin agents with click-to-approve,"Form-filling agents (school, insurance)",Trip-planning agent
4,"Full suite (docs, storage, budgeting)",Vault / on-device sensitive mode,Autopilot / recurring tasks,Email + calendar integrations
5,Household seats (up to 5),Priority / advanced-model access,Trip-planning agent,Subscription and bill tracking
6,Usage allowance (Light / Standard / Max),No-training guarantee,Referral or household gift seats,"Form-filling agents (school, insurance)"
7,"Form-filling agents (school, insurance)",Priority / advanced-model access,Life-admin agents with click-to-approve,Email + calendar integrations
8,Household seats (up to 5),Life-admin agents with click-to-approve,Subscription and bill tracking,Autopilot / recurring tasks
9,Usage allowance (Light / Standard / Max),"Full suite (docs, storage, budgeting)",Referral or household gift seats,Email + calendar integrations


In [4]:
template = []
for set_id, row in maxdiff_design.iterrows():
    template.append(
        {
            "respondent_id": "R01",
            "occupation": "professional",  # professional / student / other
            "classmate": 0,
            "set_id": set_id,
            "option_A": row["A"],
            "option_B": row["B"],
            "option_C": row["C"],
            "option_D": row["D"],
            "best": "",
            "worst": "",
        }
    )
template_df = pd.DataFrame(template)
MAXDIFF_PATH.parent.mkdir(parents=True, exist_ok=True)
if not MAXDIFF_PATH.exists():
    template_df.to_csv(MAXDIFF_PATH, index=False)
    print(f"Wrote empty template → {MAXDIFF_PATH}")
    print("Duplicate the R01 block for R02…R15, fill best/worst with the exact item text, re-run this notebook.")
else:
    print(f"Found {MAXDIFF_PATH} — scoring with whatever is already filled in.")
template_df.head(4)


Found Data/maxdiff_responses.csv — scoring with whatever is already filled in.


,respondent_id,occupation,classmate,set_id,option_A,option_B,option_C,option_D,best,worst
0,R01,professional,0,1,Autopilot / recurring tasks,No-training guarantee,Priority / advanced-model access,Subscription and bill tracking,,
1,R01,professional,0,2,Vault / on-device sensitive mode,"Full suite (docs, storage, budgeting)",Referral or household gift seats,Usage allowance (Light / Standard / Max),,
2,R01,professional,0,3,Household seats (up to 5),Life-admin agents with click-to-approve,"Form-filling agents (school, insurance)",Trip-planning agent,,
3,R01,professional,0,4,"Full suite (docs, storage, budgeting)",Vault / on-device sensitive mode,Autopilot / recurring tasks,Email + calendar integrations,,


### Score MaxDiff (best − worst)

If `best` / `worst` are still blank, scores stay empty and the **working grid in §2 stays at the Exhibit 8 default.** Once n ≥ 15 completes are in the CSV, this cell names who made the conjoint, who died, and which calls sit on the fence.


In [5]:
raw = pd.read_csv(MAXDIFF_PATH)
filled = raw.dropna(subset=["best", "worst"], how="any")
filled = filled[(filled["best"].astype(str).str.strip() != "") & (filled["worst"].astype(str).str.strip() != "")]
n_resp = filled["respondent_id"].nunique() if len(filled) else 0
print(f"Completed MaxDiff respondents: {n_resp}  (need ≥ 15)")

if n_resp == 0:
    scores = pd.DataFrame(
        {"item": ITEMS, "best": 0, "worst": 0, "appearances": appearances.reindex(ITEMS).values,
         "bw": np.nan, "bw_norm": np.nan}
    )
    made, died, borderline = [], [], []
else:
    if n_resp < 15:
        print("WARNING: n < 15. Do not file the memo on this cut.")
    counts = []
    for item in ITEMS:
        shown = (
            (filled["option_A"] == item)
            | (filled["option_B"] == item)
            | (filled["option_C"] == item)
            | (filled["option_D"] == item)
        )
        counts.append(
            {
                "item": item,
                "best": int((filled["best"] == item).sum()),
                "worst": int((filled["worst"] == item).sum()),
                "appearances": int(shown.sum()),
            }
        )
    scores = pd.DataFrame(counts)
    scores["bw"] = scores["best"] - scores["worst"]
    scores["bw_norm"] = scores["bw"] / scores["appearances"].clip(lower=1)
    scores = scores.sort_values("bw_norm", ascending=False).reset_index(drop=True)

    # Folded attributes: an Exhibit 8 slot "makes it" if ANY of its long-list items rank in the top half.
    slot_best = (
        scores.merge(longlist, left_on="item", right_on="item")
        .groupby("maps_to")["bw_norm"]
        .max()
        .sort_values(ascending=False)
    )
    print("\nBest item in each Exhibit 8 slot:")
    print(slot_best.round(3).to_string())

scores


Completed MaxDiff respondents: 0  (need ≥ 15)


,item,best,worst,appearances,bw,bw_norm
0,Usage allowance (Light / Standard / Max),0,0,4,NaN,NaN
1,No-training guarantee,0,0,4,NaN,NaN
2,Vault / on-device sensitive mode,0,0,4,NaN,NaN
3,Life-admin agents with click-to-approve,0,0,3,NaN,NaN
4,Autopilot / recurring tasks,0,0,3,NaN,NaN
5,Email + calendar integrations,0,0,4,NaN,NaN
6,"Full suite (docs, storage, budgeting)",0,0,4,NaN,NaN
7,Household seats (up to 5),0,0,4,NaN,NaN
8,Priority / advanced-model access,0,0,3,NaN,NaN
9,"Form-filling agents (school, insurance)",0,0,4,NaN,NaN


In [6]:
if n_resp >= 15:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    plot = scores.sort_values("bw_norm")
    ax.barh(plot["item"], plot["bw_norm"], color="#4C72B0")
    ax.axvline(0, color="black", linewidth=0.6)
    ax.set_xlabel("MaxDiff score (best − worst) / appearances")
    ax.set_title(f"MaxDiff counting analysis · n = {n_resp}")
    fig.tight_layout()
else:
    print("No chart until MaxDiff responses are in Data/maxdiff_responses.csv.")


No chart until MaxDiff responses are in Data/maxdiff_responses.csv.


## 2. Final attribute grid vs Exhibit 8

**Rules (Appendix 5 / Exhibit 8):** keep price, usage, and privacy in some form; adapt or swap the rest with justification; ≤ 19 non-price levels; levels concrete and buildable.

**Working grid (launch this week):** the canonical six-attribute design. Week 2 telemetry supports every slot. MaxDiff can still kill a borderline *level family* before Discover goes live — if household finishes last by a wide margin, swap it for a folded life-admin item and re-check the level cap.

Folding (judgment, not MaxDiff): trip-planning, form-filling, bills, and referral/gift seats do **not** get their own attributes. They are too granular for 5–6 slots and would blow the 19-level cap. They become concrete wording *inside* agents / integrations / seats.


In [7]:
grid = pd.DataFrame(
    [
        ["Usage allowance", "Light (60 msgs/day, fast model)",
         "Standard (unlimited fast, 25 advanced runs/mo)",
         "Max (unlimited advanced, priority)"],
        ["Privacy", "Standard (chats may train models)",
         "No-training guarantee",
         "Vault (no-training + on-device sensitive mode)"],
        ["Assistant scope", "Answers only",
         "Life-admin agents (approve each action)",
         "Autopilot (recurring tasks, approval rules)"],
        ["Integrations", "None",
         "Email + calendar",
         "Full suite (email, calendar, docs, storage, budgeting)"],
        ["Household seats", "Individual",
         "Household (up to 5)",
         ""],
        ["Price / month", "$6", "$12", "$20 / $28 / $36"],
    ],
    columns=["attribute", "level_1", "level_2", "level_3_or_more"],
)

nonprice = grid[grid["attribute"] != "Price / month"]
n_levels = int(
    nonprice[["level_1", "level_2", "level_3_or_more"]]
    .map(lambda x: str(x).strip() != "")
    .to_numpy()
    .sum()
)
# Price has 5 levels; the cell above collapsed $20/$28/$36 for display.
PRICE_LEVELS = ["$6", "$12", "$20", "$28", "$36"]
checks = pd.Series(
    {
        "has_price": True,
        "has_usage": True,
        "has_privacy": True,
        "n_attributes_ex_price": 5,
        "n_nonprice_levels": n_levels,
        "nonprice_cap_19": n_levels <= 19,
        "price_levels": len(PRICE_LEVELS),
        "levels_concrete": True,
    }
)
print("Exhibit 8 compliance")
print(checks.to_string())
print(f"\nPrice levels: {PRICE_LEVELS}")
grid


Exhibit 8 compliance
has_price                True
has_usage                True
has_privacy              True
n_attributes_ex_price       5
n_nonprice_levels          14
nonprice_cap_19          True
price_levels                5
levels_concrete          True

Price levels: ['$6', '$12', '$20', '$28', '$36']


,attribute,level_1,level_2,level_3_or_more
0,Usage allowance,"Light (60 msgs/day, fast model)","Standard (unlimited fast, 25 advanced runs/mo)","Max (unlimited advanced, priority)"
1,Privacy,Standard (chats may train models),No-training guarantee,Vault (no-training + on-device sensitive mode)
2,Assistant scope,Answers only,Life-admin agents (approve each action),"Autopilot (recurring tasks, approval rules)"
3,Integrations,None,Email + calendar,"Full suite (email, calendar, docs, storage, budgeting)"
4,Household seats,Individual,Household (up to 5),
5,Price / month,$6,$12,$20 / $28 / $36


### Hand-verify: design counts (Appendix 7)

One member reproduces this against Discover's design report before launch. 12 tasks × 3 concepts = **36 profiles per respondent**. A balanced design should come close to the expected counts below; if a level is missing or doubled, fix it in Discover before the pilots.


In [8]:
N_TASKS = 12
N_CONCEPTS = 3
profiles_per_resp = N_TASKS * N_CONCEPTS
level_counts = pd.DataFrame(
    [
        ["Usage allowance", 3, profiles_per_resp / 3],
        ["Privacy", 3, profiles_per_resp / 3],
        ["Assistant scope", 3, profiles_per_resp / 3],
        ["Integrations", 3, profiles_per_resp / 3],
        ["Household seats", 2, profiles_per_resp / 2],
        ["Price", 5, profiles_per_resp / 5],
    ],
    columns=["attribute", "n_levels", "expected_appearances_per_level"],
)
print(f"{N_TASKS} tasks × {N_CONCEPTS} concepts = {profiles_per_resp} profiles / respondent")
print(f"Non-price levels = {n_levels}  (cap 19) · price levels = {len(PRICE_LEVELS)} and include $20")
print("After building in Discover, paste the observed level frequencies next to expected.")
level_counts.style.format({"expected_appearances_per_level": "{:.1f}"})


12 tasks × 3 concepts = 36 profiles / respondent
Non-price levels = 14  (cap 19) · price levels = 5 and include $20
After building in Discover, paste the observed level frequencies next to expected.


,attribute,n_levels,expected_appearances_per_level
0,Usage allowance,3,12.0
1,Privacy,3,12.0
2,Assistant scope,3,12.0
3,Integrations,3,12.0
4,Household seats,2,18.0
5,Price,5,7.2


### Wording audit (double-barreled / overlapping)

AI is expected to red-team this; a human still has to agree. Flags we already resolved:

| Level | Risk | Fix |
|---|---|---|
| Vault (no-training + on-device) | Double-barreled: two benefits in one step | Keep — Exhibit 8 defines Vault as the stacked step; WTP is for the *package* |
| Max (unlimited advanced, priority) | Two benefits | Keep — both are the usage power fence; splitting would add a 7th attribute |
| Life-admin agents (approve each action) | “Life-admin” is a bundle | Concrete on-screen examples: draft email, fill a form, track a bill — still one click-to-approve control |
| Full suite (email, calendar, docs, storage, budgeting) | List, not a single thing | Keep as the integrations top step vs email+calendar; do not also field bill-tracking as its own attribute |
| Standard privacy (chats may train) | Sounds punitive | Needed as the $20-clone level; wording stays factual |

Do not add “and faster” or “and more private” onto unrelated levels.


### Two sentences per attribute (why it earns its slot)

**Usage allowance.** Exhibit 8 requires it, and Week 2 made it load-bearing: clickers run 10.9 sessions/week vs 6.6, and the p*-eligible group runs 13.6 vs 4.8. Light / Standard / Max also absorbs “priority / advanced-model” from the long-list, so that item dies as a standalone attribute.

**Privacy (Standard / no-training / Vault).** This is Lantern’s only claimed fence vs the $20 incumbents, and `privacy_mode_share` was the largest Week-2 coefficient. Two steps (no-training, then Vault) are what let Week 4 report a privacy-step WTP — one of the three silicon-twin predictions.

**Assistant scope (Answers / life-admin / Autopilot).** Agent-preview use separated clickers (5.7 vs 3.8 uses/30d), and the case’s identity is “an assistant that asks permission.” Autopilot is the expensive fence ($4.20 serving); form-filling, trip planning, and bill tracking are *examples* of life-admin, not their own attributes.

**Integrations (None / email+calendar / full suite).** Incumbents fence on Gmail/Docs/Office; Lantern cannot measure WTP for that fence if the attribute is missing. Email+calendar vs full suite also swallows docs/storage/budgeting and subscription tracking from the long-list.

**Household seats (Individual / Household up to 5).** Borderline. Shared-device users are 22% of MAU but clicked the banner *less* (6.0% vs 10.9%), so MaxDiff is allowed to kill this slot. It stays in the working grid because Google/Microsoft already sell family SKUs, Exhibit 8 includes it, and a two-level attribute is cheap against the 19-level cap. If MaxDiff ranks every seats item at the bottom, swap household for a two-level “referral / gift seats” only if you still want a social fence; otherwise drop it and do not replace.

**Price ($6 / $12 / $20 / $28 / $36).** Required. Firefly’s $19.99 was an imitation, not a measurement. Interpolation in simulation is legitimate; the $20 incumbent clone is a mandatory Week 5 benchmark.

### Made / died / borderline (update after MaxDiff)

| Slot | Call | What happened to the long-list |
|---|---|---|
| Usage | **Made** | Standalone “priority / advanced-model” **died** (folded into Max). |
| Privacy | **Made** | No-training and Vault **made** as two levels of one attribute. |
| Assistant scope | **Made** | Life-admin **made**; Autopilot **made** as the top step; form-filling, trip-planning **died** as attributes (worded into life-admin). |
| Integrations | **Made** | Email+calendar and full suite **made**; bill tracking **died** as a standalone. |
| Household | **Borderline — keep pending MaxDiff** | Gift/referral seats **died**. Household survives as a two-level fence unless MaxDiff last-places it. |
| Price | **Made** (not MaxDiff’d) | Always in. |


## 3. Build, pilot, and launch the CBC (Discover)

**Design rules to punch into Sawtooth Discover**

| Spec | Setting |
|---|---|
| Attributes × levels | Usage 3, Privacy 3, Agents 3, Integrations 3, Seats 2, Price 5 |
| Concepts per task | 3 paid plans + **none / dual-response** |
| Tasks per respondent | **12** (inside the 10–12 window) |
| Dual-response none | After picking the favorite of the 3 plans: “Would you actually subscribe to this plan, or keep using free Lantern?” |
| None / keep-free wording | “I would keep using the free version of Lantern (and other free tools).” |
| Prohibitions | None — every combo is buildable |
| Randomization | Default balanced overlap; fix one preview version for the silicon twin |
| Screener | Very or somewhat likely to pay in the next 12 months |
| Quotas (n ≥ 60, target 100+) | Professional 56% · Student 21% · Other 23% · classmates ≤ 20% |
| Time | ~8–10 minutes on a phone; drop speeders in Week 4 |

**What stays free (write this on the first survey screen, not as a conjoint level).** Free Lantern keeps answers, capped usage, standard privacy (chats may train), no agents, no integrations, one seat. Paid plans are upgrades from that floor. The none option is that floor.

**Live link.** Paste it here when Discover is live, and put the same URL in the milestone memo:

```
LIVE_LINK = ""   # e.g. https://...sawtoothsoftware.com/...
```


In [9]:
LIVE_LINK = ""  # paste the Discover URL once the survey is live

pilot_log = pd.DataFrame(
    [
        ["P1 — human / phone", "", "", "", ""],
        ["P2 — human / phone", "", "", "", ""],
        ["P3 — human / phone", "", "", "", ""],
        ["P4 — human / phone", "", "", "", ""],
        ["P5 — human / phone", "", "", "", ""],
        ["P6 — AI run-through", "", "", "", ""],
    ],
    columns=["pilot", "date", "minutes", "what_broke", "fix_shipped"],
)
# If you keep a filled log on disk, this cell will display it instead of the blank.
log_path = DATA_DIR / "cbc_pilot_log.csv"
if log_path.exists():
    pilot_log = pd.read_csv(log_path)
else:
    pilot_log.to_csv(log_path, index=False)
    print(f"Wrote blank pilot log → {log_path}")

print("Live link:", LIVE_LINK or "(not yet — CBC is not launched)")
print("Need 5 human pilots + 1 full AI run-through before launch.")
pilot_log


Wrote blank pilot log → Data/cbc_pilot_log.csv
Live link: (not yet — CBC is not launched)
Need 5 human pilots + 1 full AI run-through before launch.


,pilot,date,minutes,what_broke,fix_shipped
0,P1 — human / phone,,,,
1,P2 — human / phone,,,,
2,P3 — human / phone,,,,
3,P4 — human / phone,,,,
4,P5 — human / phone,,,,
5,P6 — AI run-through,,,,


**Pilot-fix checklist (what to watch for)**

- Every team member takes the survey **end to end on a phone** and logs minutes (exercise brief).
- Dual-response none actually fires after the 3-concept pick — not a hidden fourth concept.
- Price always present; no task with two identical concepts.
- Level wording fits on a phone (Vault and Max are the long ones).
- Occupations map to the quota question.
- A complete AI run-through finishes all 12 tasks without dead ends.
- Freeze **one** preview version after the last fix. That frozen instrument is what the silicon twin and the humans both see.

Once five humans and one AI pass and the fixes are in the log: **launch.** Do not wait for MaxDiff n = 15 if the working grid is still Exhibit 8 — MaxDiff can confirm household the same day. Waiting a day to launch costs a day of the seven-day fielding window.


## 4. Silicon twin (Appendix 9)

Protocol: fix one preview of the survey; build **30 personas from the telemetry**; run each through the **same 12 tasks in a fresh chat**; preregister three predictions **before** human fielding closes. Budget ~5 minutes per persona, 2.5–3 team-hours. Accuracy is not graded. Honesty is.

Personas below are a stratified draw from the Fact Pack’s 5,000-user sample, weighted toward higher predicted upgrade intent (the consideration-set analogue), occupation mix matched to the 124k pool (17 professional / 6 student / 7 other).


In [10]:
df = pd.read_excel(FACT_PACK, sheet_name="3_Free_User_Telemetry")
num_feats = [
    "tenure_months", "sessions_per_week", "avg_msgs_per_session",
    "agent_preview_uses_30d", "doc_uploads_30d", "privacy_mode_share",
    "household_shared_device", "referred_by_friend",
]
X = df[num_feats].copy()
dummies = pd.get_dummies(df[["platform", "occupation"]], drop_first=True, dtype=float)
X = sm.add_constant(pd.concat([X, dummies], axis=1).astype(float), has_constant="add")
model = sm.Logit(df[TARGET].astype(float), X).fit(disp=False)
df = df.copy()
df["p_intent"] = model.predict(X)

quota = {"professional": 17, "student": 6, "other": 7}
picks = []
for occ, n in quota.items():
    pool = df[df["occupation"] == occ]
    pick = pool.sample(n=n, weights=pool["p_intent"].clip(lower=1e-4), random_state=RNG)
    picks.append(pick)
personas = pd.concat(picks).sample(frac=1, random_state=RNG).reset_index(drop=True)
personas.insert(0, "persona_id", [f"ST{i:02d}" for i in range(1, 31)])

def brief(row):
    device = "a shared household device" if row.household_shared_device else "a single-user device"
    via = "a friend’s referral link" if row.referred_by_friend else "your own signup"
    return (
        f"You are {row.occupation} who mainly uses Lantern on {row.platform}. "
        f"You signed up {int(row.tenure_months)} months ago and now open it about "
        f"{row.sessions_per_week:.1f} times a week, sending {row.avg_msgs_per_session:.1f} messages a session. "
        f"In the last 30 days you used the free life-admin agent preview "
        f"{int(row.agent_preview_uses_30d)} times and uploaded {int(row.doc_uploads_30d)} documents. "
        f"{row.privacy_mode_share:.0%} of your chats start in private (no-history) mode. "
        f"You use {device} and arrived via {via}. "
        f"You told Lantern you are considering paying for a better assistant in the next 12 months. "
        f"Answer every survey question as this person, not as an assistant giving advice."
    )

personas["prompt_brief"] = personas.apply(brief, axis=1)
keep = [
    "persona_id", "user_id", "occupation", "platform", "tenure_months",
    "sessions_per_week", "avg_msgs_per_session", "agent_preview_uses_30d",
    "doc_uploads_30d", "privacy_mode_share", "household_shared_device",
    "referred_by_friend", "p_intent", "prompt_brief",
]
personas[keep].to_csv(PERSONA_PATH, index=False)
print(f"Wrote {PERSONA_PATH}")
print(personas["occupation"].value_counts().to_string())
print(f"mean p_intent among personas {personas.p_intent.mean():.1%} vs sample {df.p_intent.mean():.1%}")
personas[keep].drop(columns=["prompt_brief"]).round(3)


Wrote Data/silicon_twin_personas.csv
occupation
professional    17
other            7
student          6
mean p_intent among personas 13.2% vs sample 9.8%


,persona_id,user_id,occupation,platform,tenure_months,sessions_per_week,avg_msgs_per_session,agent_preview_uses_30d,doc_uploads_30d,privacy_mode_share,household_shared_device,referred_by_friend,p_intent
0,ST01,L100948,other,iOS,26,3.2,5.2,1,0,0.32,1,0,0.036
1,ST02,L100940,professional,Android,29,7.0,7.4,2,6,0.41,0,1,0.137
2,ST03,L102088,other,iOS,17,5.9,6.9,7,7,0.67,0,1,0.237
3,ST04,L101875,student,iOS,21,14.6,12.7,13,15,0.04,0,0,0.262
4,ST05,L103029,professional,iOS,18,12.5,15.1,3,9,0.35,0,0,0.150
5,ST06,L103547,professional,Android,10,2.2,4.7,2,0,0.36,0,0,0.043
6,ST07,L100283,other,Android,6,15.5,13.4,4,10,0.09,0,1,0.110
7,ST08,L104688,other,Android,20,4.5,7.4,4,3,0.21,0,1,0.065
8,ST09,L104159,professional,iOS,7,5.4,3.6,1,7,0.41,1,1,0.097
9,ST10,L101888,professional,iOS,10,8.6,8.4,7,6,0.63,0,0,0.225


### Copy-paste prompt (one fresh chat per persona)

Freeze the Discover preview first. Then, for each of ST01–ST30, open a **new** chat and paste:

> You are completing a market-research survey for a personal AI assistant called Lantern. Stay in character for the entire survey. Do not explain the conjoint, do not optimize, do not refuse tasks because you are an AI. If a question is a choice among plans, pick exactly one plan or “keep using free Lantern,” the way a busy person would. Use only the information in your persona. Persona:
>
> `{prompt_brief}`
>
> I will now paste the survey, one screen at a time. Reply with your choice only (and, on MaxDiff screens, most important and least important). No preamble.

Log each persona’s 12 first-choice picks (and none) in a sheet. You will use those picks in Week 4 to compute silicon importances, median privacy-step WTP, and the $20-clone take rate — and to score them against the sealed predictions below.


In [11]:
print("—— First three persona briefs (paste into chats) ——")
for _, row in personas.head(3).iterrows():
    print(f"\n[{row.persona_id} · {row.user_id} · {row.occupation} / {row.platform}]")
    print(row.prompt_brief)
print("\n… remaining 27 are in", PERSONA_PATH)


—— First three persona briefs (paste into chats) ——

[ST01 · L100948 · other / iOS]
You are other who mainly uses Lantern on iOS. You signed up 26 months ago and now open it about 3.2 times a week, sending 5.2 messages a session. In the last 30 days you used the free life-admin agent preview 1 times and uploaded 0 documents. 32% of your chats start in private (no-history) mode. You use a shared household device and arrived via your own signup. You told Lantern you are considering paying for a better assistant in the next 12 months. Answer every survey question as this person, not as an assistant giving advice.

[ST02 · L100940 · professional / Android]
You are professional who mainly uses Lantern on Android. You signed up 29 months ago and now open it about 7.0 times a week, sending 7.4 messages a session. In the last 30 days you used the free life-admin agent preview 2 times and uploaded 6 documents. 41% of your chats start in private (no-history) mode. You use a single-user device an

### Sealed preregistration — file before human fielding closes

Fill this cell **after** the 30 silicon runs, **before** you look at human HB estimates (you may look at silicon-only tallies). Accuracy is not graded. Changing these numbers after seeing humans is a failing answer.

The three predictions:

1. **Top-2 attributes by importance** (silicon HB or, if you only have first choices, the two attributes that most often swing the pick).
2. **Median WTP for the privacy step** — Vault vs Standard, $/month, using the Exhibit 10 recipe: `(u_Vault − u_Standard) / |β_$10| × 10`.
3. **Take rate of a $20 clone** — first-choice share for a single paid plan: Standard usage, Standard privacy (chats may train), Answers only, no integrations, Individual, $20, versus None.


In [12]:
from datetime import datetime, timezone

# Fill after the 30 silicon runs. Leave None until then.
PREREG = {
    "timestamp_utc": None,  # e.g. datetime.now(timezone.utc).isoformat(timespec="minutes")
    "top2_attributes": [None, None],  # e.g. ["Privacy", "Usage allowance"]
    "median_privacy_step_wtp_usd": None,  # Vault vs Standard, $/month
    "clone20_take_rate": None,  # 0–1, first-choice vs None
    "n_silicon": 30,
    "instrument_version": "Discover preview frozen at launch",
    "notes": "",
}

if PREREG["timestamp_utc"] is None:
    print("UNSEALED — silicon twin not yet scored. Do not backfill after human HB.")
else:
    print("SEALED at", PREREG["timestamp_utc"])
pd.Series(PREREG, dtype=object).to_frame("value")


UNSEALED — silicon twin not yet scored. Do not backfill after human HB.


,value
timestamp_utc,None
top2_attributes,"[None, None]"
median_privacy_step_wtp_usd,None
clone20_take_rate,None
n_silicon,30
instrument_version,Discover preview frozen at launch
notes,


## Memo copy-paste

**1. MaxDiff.** Fielded the 13 non-price items from Week 2 in 12 sets of 4. n = ___ (need ≥ 15). **Made:** usage, privacy, assistant scope, integrations, [household if not last]. **Died as attributes:** priority/advanced-model (folded into Max); form-filling, trip-planning, bill tracking (worded into life-admin / full suite); referral/gift seats. **Borderline:** household seats — shared-device users click less in telemetry; kept as a two-level fence unless MaxDiff last-places it.

**2. Grid vs Exhibit 8.** Five attributes + price, 14 non-price levels (≤ 19). Price, usage, and privacy kept. Two sentences per attribute are in §2. None option: “keep using the free version of Lantern.”

**3. CBC.** Discover: 12 tasks, 3 concepts + dual-response none. Five human phone pilots + one AI run-through logged in `Data/cbc_pilot_log.csv`. Live link: ___ . Free tier stated on the intro screen: capped answers, standard privacy, no agents, no integrations, one seat. Design counts: 12 × 3 = 36 profiles/respondent; 14 non-price levels.

**4. Silicon twin.** 30 personas drawn from Fact Pack telemetry (17/6/7 occupation mix, intent-weighted) in `Data/silicon_twin_personas.csv`. Sealed predictions filed at ___ : top-2 = ___ ; median Vault-vs-Standard WTP = $___ ; $20-clone take rate = ___ . Not opened against human data.
